# Lab 5 — Build an OpenAI-Compatible API
**Day 2 Morning | ~45 minutes | Colab CPU | `OPENAI_API_KEY` + `NGROK_AUTH_TOKEN`**

---

Since Lab 1A you have been the client. You built an `OpenAI(...)` object, pointed it at someone else's URL, and text came back. Today you write what sits behind the URL.

It is a small server: five short cells of FastAPI, about fifty lines. It does not run a model itself. It forwards each request to `gpt-4o-mini` and hands the answer back in the OpenAI format. That sounds like a toy, and the point of the lab is that it isn't one. vLLM, Ollama and TGI expose this same shape, so the day a real model sits behind your server, the client code you write this morning does not change.

## What you will walk out with

1. An LLM API is an HTTP route that takes JSON and returns JSON. You will write all of it.
2. "OpenAI-compatible" means a request schema and a response schema. Match them and every OpenAI client can call you.
3. Streaming is an ordinary HTTP pattern (Server-Sent Events). You will read the raw lines.
4. One client, several backends: your server, OpenAI, and optionally Groq, with only `base_url` changing.
5. What your server cannot do once a real model lives behind it, and what vLLM does instead.

## The route

```
Your Python code (OpenAI client)
        │  base_url = your ngrok URL (or localhost)
        ▼
YOUR FastAPI server, port 8000 in this runtime
        │  forwards the request
        ▼
OpenAI today  ·  vLLM on a GPU in production
```

The client at the top never learns which box is at the bottom. Every cell below is one arrow in that picture.

| Tool | Its job here |
|---|---|
| **FastAPI** | Declares the routes (`/health`, `/v1/chat/completions`) and the JSON each one accepts |
| **uvicorn** | The process that listens on port 8000 and hands each request to FastAPI |
| **ngrok** | Colab has no public address. ngrok gives port 8000 an HTTPS URL anyone can reach |

**Coming from Lab 4:** you trained an adapter on a GPU. This notebook does not serve it, because Colab's CPU runtime cannot run vLLM. What you build here is the part that stays the same when you do.

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} fastapi uvicorn pyngrok openai httpx python-dotenv

### What you just installed

- `fastapi` and `uvicorn`: the framework, and the process that runs it.
- `pyngrok`: opens the public tunnel from Python.
- `openai`: used on both sides of the wire. The server uses it to call OpenAI; your client code uses it to call the server.
- `httpx`: plain HTTP requests, for poking the server without the SDK in the way.

In [ ]:
import os
try:
    from google.colab import userdata          # Colab: read the Secrets you added
    os.environ["OPENAI_API_KEY"]   = userdata.get("OPENAI_API_KEY")
    os.environ["NGROK_AUTH_TOKEN"] = userdata.get("NGROK_AUTH_TOKEN")
except ImportError:
    from dotenv import load_dotenv             # local: read .env in the repo root
    load_dotenv()
assert os.environ.get("OPENAI_API_KEY"),   "Add OPENAI_API_KEY as a Colab Secret or to .env"
assert os.environ.get("NGROK_AUTH_TOKEN"), "Add NGROK_AUTH_TOKEN as a Colab Secret or to .env"

OPENAI_API_KEY   = os.environ["OPENAI_API_KEY"]
NGROK_AUTH_TOKEN = os.environ["NGROK_AUTH_TOKEN"]
OPENAI_BASE_URL  = "https://api.openai.com/v1"
DEFAULT_MODEL    = "gpt-4o-mini"
print(f"Ready — {DEFAULT_MODEL}, ngrok token found")

---

## FastAPI in one example

Here is an OpenAI-shaped server with everything useful taken out:

```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class ChatRequest(BaseModel):      # the JSON body this route accepts
    model: str
    messages: list

@app.post("/v1/chat/completions")  # run this function when a POST arrives at this path
def chat(req: ChatRequest):
    return {"choices": [...]}
```

Three lines do the work:

- `app = FastAPI()` is the application object. uvicorn runs it.
- `class ChatRequest(BaseModel)` is a Pydantic schema. Send a body that does not fit and FastAPI answers `422` before your function runs. You will trigger that on purpose in Part B.
- `@app.post(...)` registers the function for that path. FastAPI also reads these decorators to build a `/docs` page for free.

That is all the OpenAI API is from the outside: a few paths, a request shape, a response shape. Match the shapes and the OpenAI client cannot tell you apart.

### How to read the server cells

`server.py` is written in **five short cells**. The first creates the file (`%%writefile server.py`); the next four append to it (`%%writefile -a`). Run them top to bottom once. If you edit one, re-run all five from A1.1.

| Cell | What it adds | Why it matters |
|---|---|---|
| A1.1 | app object + backend config from env vars | Swap the backend by changing env vars, not code |
| A1.2 | Pydantic request schemas | Wrong JSON is rejected before your code runs |
| A1.3 | `/health`, `/v1/models` | What load balancers and clients probe |
| A1.4 | `/v1/chat/completions` | The OpenAI request and response format |
| A1.5 | the streaming path | Server-Sent Events: one `data:` line per piece |

Today the server forwards to OpenAI. Point `BACKEND_BASE_URL` at a vLLM server and nothing else in the file changes.

---

# Part A — Write and launch the server

**~20 minutes**

A web server is a process that never finishes. It sits on a port and waits. Start one inside a notebook cell and that cell runs forever; you could not run anything else. So we split the job:

1. Write the server code into a file, `server.py`, from notebook cells.
2. Start that file as a separate process with `subprocess.Popen`, which returns immediately.
3. Keep using the notebook to call the server while it runs next door.

The process stays alive until the cleanup cell at the end. Re-run the launch cell without cleaning up and you get "port 8000 already in use": run cleanup, then launch again.

The launch command is `uvicorn server:app --host 127.0.0.1 --port 8000`. In Lab 10's Dockerfile it is the same command with one change, `--host 0.0.0.0`. Here, the only thing that needs to reach port 8000 is ngrok, running on the same machine, so listening on `127.0.0.1` is enough. Inside a container, requests arrive from outside the container, so the server has to listen on every interface. Same file, same routes, one flag.

### A1.1 — App object and backend config

In [ ]:
%%writefile server.py
import os, json, time, uuid
from typing import List, Optional
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from openai import OpenAI

app = FastAPI(title="My LLM API", version="0.1.0")

# Which backend this server forwards to. Change the env vars and restart; nothing else changes.
BACKEND_API_KEY  = os.environ.get("BACKEND_API_KEY", "")
BACKEND_BASE_URL = os.environ.get("BACKEND_BASE_URL", "https://api.openai.com/v1")
DEFAULT_MODEL    = os.environ.get("DEFAULT_MODEL", "gpt-4o-mini")

backend = OpenAI(api_key=BACKEND_API_KEY, base_url=BACKEND_BASE_URL)

### A1.2 — Request schemas

Pydantic models define the JSON the server accepts. If a caller sends the wrong shape, FastAPI returns `422` before your function runs. These two classes *are* the OpenAI request format. `-a` appends to the file.

In [ ]:
%%writefile -a server.py

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str = DEFAULT_MODEL
    messages: List[Message]
    stream: bool = False
    temperature: float = 0.7
    max_tokens: Optional[int] = 500

### A1.3 — Two small routes

`/health` is what a load balancer polls. `/v1/models` is what an OpenAI client can list. Each is a plain function with a decorator.

In [ ]:
%%writefile -a server.py

@app.get("/health")
def health():
    return {"status": "ok", "backend": BACKEND_BASE_URL, "model": DEFAULT_MODEL}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": DEFAULT_MODEL, "object": "model", "owned_by": "custom-server"}]}

### A1.4 — The main route

`POST /v1/chat/completions` is the one route that does real work. Start with the simple path: forward the messages to the backend, wait for the whole answer, and send it back in the OpenAI response shape (an `id`, the `choices` list with one assistant message, and the token `usage`).

Streaming requests take a different path. One line at the top of the route hands them to a function called `sse_stream`, which the next cell writes.

In [ ]:
%%writefile -a server.py

@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest):
    msgs = [{"role": m.role, "content": m.content} for m in req.messages]
    cid = f"chatcmpl-{uuid.uuid4().hex[:8]}"          # our own response id

    if req.stream:
        return StreamingResponse(sse_stream(req, msgs, cid), media_type="text/event-stream")

    resp = backend.chat.completions.create(model=req.model, messages=msgs,
                                           temperature=req.temperature, max_tokens=req.max_tokens)
    return {
        "id": cid,
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{"index": 0, "finish_reason": "stop",
                     "message": {"role": "assistant", "content": resp.choices[0].message.content}}],
        "usage": resp.usage.model_dump(),
    }

### A1.5 — The streaming path

A streaming response is not one JSON body. It is a connection that stays open while the server writes one line per piece of the answer, in the **Server-Sent Events** format:

```
data: {"id": "chatcmpl-1a2b3c4d", "object": "chat.completion.chunk", "choices": [{"delta": {"content": "Hel"}}]}

data: {"id": "chatcmpl-1a2b3c4d", "object": "chat.completion.chunk", "choices": [{"delta": {"content": "lo"}}]}

data: [DONE]
```

Each line starts with `data: `, is followed by a blank line, and the last one is `data: [DONE]`. That is the whole format.

`sse_stream` is a generator (Lab 7 opens up `yield` properly). It asks the backend to stream, and for every piece of text that arrives it yields one `data:` line. FastAPI's `StreamingResponse` sends each line the moment it is yielded. That is why ChatGPT looks like it is typing.

In [ ]:
%%writefile -a server.py

def sse_stream(req, msgs, cid):
    stream = backend.chat.completions.create(model=req.model, messages=msgs,
                                             temperature=req.temperature, stream=True)
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            data = {"id": cid, "object": "chat.completion.chunk", "created": int(time.time()),
                    "model": req.model,
                    "choices": [{"index": 0, "delta": {"content": delta}, "finish_reason": None}]}
            yield f"data: {json.dumps(data)}\n\n"
    yield "data: [DONE]\n\n"

**Checkpoint:** five cells, one file. Print it to see the whole server in one place. This is the file a container would run (Lab 10).

In [ ]:
print(open("server.py").read())

### Step A2 — Launch the server

This cell starts uvicorn in the background on `127.0.0.1:8000` and polls `GET /health` until it answers. Logs go to `uvicorn.log` (never a stdout pipe — a full pipe freezes the server on Colab). If health never comes up, open `uvicorn.log`.

Step A3 then opens an ngrok tunnel with `pyngrok` (still [ngrok's official Colab path](https://ngrok.com/docs/using-ngrok-with/googleColab)). You need a free account and the `NGROK_AUTH_TOKEN` secret. Gradio (Lab 7) uses its own `share=True` tunnel instead.

**Colab / free-plan quirks (read once):**

| What you will see | What to do |
|---|---|
| Browser interstitial “Visit Site” on the ngrok URL | Click **Visit Site**. That warning is for *browsers*. Our Python client sends `ngrok-skip-browser-warning` so API calls skip it. |
| `ERR_NGROK_107` / invalid authtoken | Check the `NGROK_AUTH_TOKEN` secret. Copy the token from the ngrok dashboard again. |
| Health check on localhost fails | Port 8000 did not come up. Open `uvicorn.log`, or run the cleanup cell and retry A2. |
| Tunnel fails but localhost health is OK | Keep going: B1–B3 use `http://127.0.0.1:8000`. You still built an OpenAI-compatible server. |
| Port 8000 already in use | Cleanup cell, then A2 again. |

After a successful tunnel: open **Swagger** (`/docs`) on your phone. That is the “this is a real API” moment.

In [ ]:
import subprocess, time, httpx, sys

LOCAL_URL = "http://127.0.0.1:8000"
env = {**os.environ, "BACKEND_API_KEY": OPENAI_API_KEY, "BACKEND_BASE_URL": OPENAI_BASE_URL, "DEFAULT_MODEL": DEFAULT_MODEL}

# Logs go to a file. Never PIPE stdout on Colab: a full pipe freezes uvicorn.
log = open("uvicorn.log", "w")
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT, env=env,
)

healthy = False
for _ in range(20):                       # up to 10 seconds
    try:
        httpx.get(f"{LOCAL_URL}/health", timeout=1).raise_for_status()
        healthy = True
        break
    except Exception:
        time.sleep(0.5)

assert healthy, "Server did not start. Open uvicorn.log for the reason."
print("Local health OK —", LOCAL_URL)

**Checkpoint:** `Local health OK`. The server is alive inside this runtime, but nothing outside can reach port 8000 yet.

### Step A3 — Open the public tunnel

ngrok gives the local port an HTTPS URL. If the tunnel fails (bad token, network policy), the lab continues on localhost — the OpenAI-compatible server is the lesson, the public URL is the demo.

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
NGROK_HEADERS = {"ngrok-skip-browser-warning": "true"}   # free plan shows a browser warning page; API calls skip it

try:
    SERVER_URL = ngrok.connect(8000).public_url
    print("Public URL :", SERVER_URL)
    print("Swagger UI :", f"{SERVER_URL}/docs")
    print("If a browser shows an ngrok warning page, click Visit Site.")
except Exception as e:
    SERVER_URL = LOCAL_URL
    print("ngrok tunnel did not open:", e)
    print("Continuing on", SERVER_URL)

#### ✅ Part A checkpoint
- [ ] `server.py` is on disk, and you have read it top to bottom.
- [ ] `Local health OK` printed.
- [ ] If the tunnel opened, the Swagger link shows your three routes. Try `/v1/models` from the page.

---

# Part B — Call your server

**~15 minutes**

The server is up. Now prove it speaks the protocol, which means: code written for OpenAI works against it unchanged.

- **B1** — plain HTTP, no SDK: a health check, then a request that should fail.
- **B2** — the OpenAI SDK pointed at your URL.
- **B3** — the same client with `stream=True`, then the raw stream without the SDK.
- **B4** — several backends from one loop.

If ngrok failed, `SERVER_URL` is `http://127.0.0.1:8000`. That is enough to prove the protocol. The public URL only matters when something outside this runtime, your phone or a classmate, wants to call you.

Leave the server running until the cleanup cell at the end.

### B1 — Health check

In [ ]:
import httpx

def get_json(url: str):
    headers = NGROK_HEADERS if "ngrok" in url else None
    r = httpx.get(url, headers=headers, timeout=15)
    r.raise_for_status()
    return r.json()

print("Local  :", get_json(f"{LOCAL_URL}/health"))
if SERVER_URL != LOCAL_URL:
    print("Public :", get_json(f"{SERVER_URL}/health"))
else:
    print("Public : (no tunnel — using localhost)")


**Checkpoint:** you see `{'status': 'ok', ...}` at least once. That dict is the return value of the `health()` function you wrote in A1.3.

Now send the server something that is not a chat request. `messages` should be a list of role/content dicts; we will send a plain string. Before you run it: does your `chat_completions` function run at all?

In [ ]:
bad = httpx.post(f"{LOCAL_URL}/v1/chat/completions", json={"messages": "hello"}, timeout=15)

print("Status:", bad.status_code)
print("Why   :", bad.json()["detail"][0]["msg"])

It never ran. FastAPI checked the body against `ChatRequest` from A1.2, saw that `messages` was not a list, and answered `422` on its own. No call to OpenAI, no tokens spent. Checking the request at the door is one of the things a real API gives you that a notebook function does not.

### B2 — The OpenAI client, pointed at you

This is the SDK and the call you used in Lab 1A. One argument is different:

```python
my_client = OpenAI(base_url=f"{SERVER_URL}/v1", api_key="not-needed")
```

`api_key="not-needed"` works because your server never checks it. It uses its own key to call OpenAI on the caller's behalf. That is a real gap: anyone who has your ngrok URL is spending your OpenAI credit. Stretch goal 2 closes it.

`make_client` below adds one header when the URL is an ngrok one, to skip the free plan's browser warning page. Otherwise it is the stock client.

Before you run it, look back at A1.4. Your server writes the response `id` itself. Will it look like an OpenAI id?

In [ ]:
from openai import OpenAI

def make_client(base_url: str, api_key: str = "not-needed"):
    """Same OpenAI SDK. Extra header only when talking through free ngrok."""
    kwargs = {"base_url": base_url, "api_key": api_key}
    if "ngrok" in base_url:
        kwargs["default_headers"] = NGROK_HEADERS
    return OpenAI(**kwargs)

my_client = make_client(f"{SERVER_URL}/v1")

resp = my_client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[
        {"role": "system", "content": "You are an LLM deployment expert."},
        {"role": "user", "content": "What are 3 advantages of vLLM over a naive FastAPI server?"},
    ],
)
print(resp.choices[0].message.content)
print()
print("id    :", resp.id)
print("tokens:", resp.usage)

**Checkpoint:** the `id` is `chatcmpl-` plus eight hex characters, which is exactly what `uuid.uuid4().hex[:8]` in A1.4 produces. OpenAI's own ids are much longer. The words came from OpenAI; the envelope came from you, and the SDK accepted it without complaint.

Read the answer itself with some suspicion. `gpt-4o-mini` is fluent about vLLM and not always right about it. Part C gives you enough to check.

### B3 — Streaming

Without streaming, the server waits for the whole answer and sends it in one piece. With streaming, it sends each piece as it arrives, using **Server-Sent Events (SSE)**: the HTTP connection stays open, and the server writes one `data: {...}` line per chunk until a final `data: [DONE]`.

Your A1.5 cell does exactly that: it calls the backend with `stream=True`, wraps each delta in a `chat.completion.chunk` JSON object, and yields it as a `data:` line.

On the client side, `stream=True` makes the SDK return an iterator. Each `chunk.choices[0].delta.content` is a few characters of text, and `print(..., end="", flush=True)` puts them on screen as they land.

In [ ]:
# Cell B3 — Streaming from your server
print('Streaming from our server:')
print('─' * 60)
stream = my_client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{'role': 'user', 'content': 'List 5 things that can go wrong when serving LLMs. Be brief.'}],
    stream=True
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print('\n' + '─' * 60)

The SDK hid the wire format. Here is the same kind of request with plain `httpx`, printing each line exactly as your server sends it:

In [ ]:
body = {"messages": [{"role": "user", "content": "Say hello in five words."}], "stream": True}

with httpx.stream("POST", f"{LOCAL_URL}/v1/chat/completions", json=body, timeout=30) as r:
    for line in r.iter_lines():
        if line:
            print(line)

**Checkpoint:** one `data:` line per chunk, each a complete JSON object carrying a few characters in `delta.content`, the same `id` on every line, and `data: [DONE]` at the end. Nothing LLM-specific about it. Any HTTP client in any language can read this, which is why every chat UI you have used streams this way.

### B4 — One loop, several backends

The loop below asks each backend the same question with the same code. Per backend, three values change: `base_url`, `api_key` and `model`.

- your server, which forwards to OpenAI
- OpenAI directly
- Groq running `openai/gpt-oss-20b`, if a `GROQ_API_KEY` secret exists

The first two are the same model reached by two routes, so the answers should be close. Groq is a different model on different hardware. Watch the timings as well as the text. LiteLLM (Bonus 04), OpenRouter and every other LLM gateway are built on this pattern.

Keep the answers on screen. Once you have read Part C, come back and grade them. Fluent is not the same as correct, and a one-sentence definition of PagedAttention is a good place to see the difference.

In [ ]:
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:                      # not on Colab, or no such secret
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

providers = {
    "Our FastAPI → OpenAI":        {"base_url": f"{SERVER_URL}/v1", "api_key": "not-needed", "model": DEFAULT_MODEL},
    "OpenAI direct (gpt-4o-mini)": {"base_url": OPENAI_BASE_URL, "api_key": OPENAI_API_KEY, "model": DEFAULT_MODEL},
}
if GROQ_API_KEY:
    providers["Groq gpt-oss-20b"] = {
        "base_url": "https://api.groq.com/openai/v1", "api_key": GROQ_API_KEY, "model": "openai/gpt-oss-20b"}
else:
    print("No GROQ_API_KEY — comparing your server vs OpenAI only.\n")

question = "In one sentence: what is PagedAttention?"
for name, cfg in providers.items():
    c = make_client(cfg["base_url"], cfg["api_key"])
    t0 = time.perf_counter()
    r = c.chat.completions.create(model=cfg["model"], messages=[{"role": "user", "content": question}])
    print(f"[{name}]  {time.perf_counter() - t0:.1f}s\n  {r.choices[0].message.content}\n")

---

# Part C — What your server cannot do

**~10 minutes, reading**

Right now your server is a thin layer over OpenAI. The expensive part, running the model, happens on OpenAI's GPUs. FastAPI runs each plain `def` handler in a thread pool, so while one request waits on the network, the next one starts. Ten classmates hitting your URL at once would be fine.

Now swap OpenAI for a model on your own T4, loaded the way you loaded it in Lab 4, with `model.generate()` inside the handler. Three things go wrong.

**Users queue for the GPU.** `generate()` works on whatever batch you hand it, and each request arrives as a batch of one. You could collect eight requests and run them together, but then all eight wait for the longest answer to finish. **Continuous batching** solves this one decoding step at a time: after every token, finished sequences leave the batch and waiting ones join. The GPU never idles behind the slowest request.

**The KV cache runs out of room before the GPU runs out of compute.** You measured the cache in Lab 3: it grows with every token of every active conversation. A naive server reserves space for the maximum length up front, and most requests are shorter, so most of the reservation sits empty. The vLLM paper found existing systems wasting 60 to 80 % of their KV cache memory this way. **PagedAttention** stores the cache in small blocks allocated as each sequence grows, the way an operating system pages memory. Waste drops under 4 %, and more users fit on the same card.

**4-bit weights need fast kernels.** In Lab 4, NF4 was slower than FP16 because every matmul dequantized first. vLLM ships fused kernels for formats such as AWQ and GPTQ, so the memory saving does not cost you the speed.

How much does it add up to? When vLLM came out in 2023, its authors measured up to 24× the throughput of plain Hugging Face Transformers, and 2 to 4× over the best serving systems of the time. The multiple depends on the model and the traffic. The order of magnitude is why nobody serves real users from a bare `generate()` loop.

And the part that matters to you: vLLM speaks the same `/v1/chat/completions` you just built.

```bash
vllm serve Qwen/Qwen2.5-7B-Instruct-AWQ --host 0.0.0.0 --port 8000
```

Run that on a GPU machine, set your server's `BACKEND_BASE_URL` to `http://that-machine:8000/v1`, and not one line of Part B changes.

You can do this for real: [Bonus 03](../Bonus/03_vllm_serving.ipynb) runs vLLM on a free Colab T4, serves your Lab 4 fine-tune, and measures continuous batching by sending 32 requests at once.

| | Your FastAPI server | vLLM |
|---|---|---|
| `/v1/chat/completions` and SSE streaming | yes | yes |
| Size of the code | ~50 lines, all yours | a large engine you configure |
| Many users on one GPU | one `generate()` at a time | continuous batching |
| KV cache memory | reserved per request | PagedAttention |
| Quantized weights | whatever the backend does | AWQ, GPTQ and FP8 kernels |
| Where it runs | anywhere Python runs, including this CPU | a GPU, in practice |

These are not rivals. Your server is where custom logic goes: auth, logging, routing, and the extras in Labs 8 to 11. vLLM is where the model runs. A common production setup is both, your FastAPI in front and vLLM behind it, which is the shape of this lab with OpenAI swapped out.

#### ✅ Checkpoint

Answer these before you clean up:

1. What is the only line that changed between calling OpenAI and calling your server?
2. What does `stream=True` change about what the server sends back?
3. If the handler called `generate()` on a T4 instead of forwarding to OpenAI, what happens when 100 users arrive in the same second?
4. What would you change in cell A2 to point the server at a vLLM instance?

<details>
<summary>Answers</summary>

1. `base_url` (and the key, which your server ignores).
2. The connection stays open and the server writes one `data:` line per chunk, ending with `data: [DONE]`, instead of one JSON body at the end.
3. They queue. Each waits for the requests ahead of it to finish generating, and the KV cache reservations run out of GPU memory long before the compute does. That is the gap continuous batching and PagedAttention close.
4. The `env` dict: set `BACKEND_BASE_URL` to the vLLM address (for example `http://your-gpu-server:8000/v1`) and `BACKEND_API_KEY` to any non-empty string, since vLLM does not check keys by default. Every line of client code in Part B stays the same.

</details>

---

## From Colab to production

Nothing in `server.py` is a notebook trick. What changes on the way to production is everything around it.

| Here | Why here | In production |
|---|---|---|
| `subprocess.Popen(uvicorn ...)` | A notebook cell cannot block forever | A container (Lab 10) or a system service running the same `uvicorn` command |
| Polling `/health` after launch | Do not call the server before it answers | A Docker `HEALTHCHECK` or load balancer probe hitting the same route |
| ngrok tunnel | Colab has no public address | A domain name, TLS, and a load balancer or reverse proxy |
| OpenAI as the backend | No GPU on this runtime | vLLM on a GPU machine, reached by changing `BACKEND_BASE_URL` |
| No key check | Keeps the lab short | Auth in front of every route (stretch goal 2) |

`server.py`, the schemas, the SSE format and every line of client code stay as they are.

---

**Run the cleanup cell now.** It stops uvicorn and closes the tunnel. Skip it and both keep running until Colab recycles the runtime.

In [ ]:
server_proc.terminate()
try:
    ngrok.kill()
except Exception:
    pass
print("Server and ngrok agent stopped.")


---

## ✅ Lab 5 complete

You ran all of this yourself:

- [ ] Wrote `server.py` in five cells and launched it with uvicorn
- [ ] Got `/health` to answer, and opened the Swagger page if the tunnel came up
- [ ] Sent a bad body and watched FastAPI reject it with `422` before your code ran
- [ ] Called your own server with the stock OpenAI client and spotted your own `id` in the reply
- [ ] Streamed through the SDK, then read the raw `data:` lines
- [ ] Ran one loop against several backends
- [ ] Said what breaks when a real model moves behind this server, and what vLLM does about it
- [ ] Ran the cleanup cell

## What to take with you

1. **An LLM API is HTTP.** A `POST` route that takes JSON and returns JSON. Any web framework can do it; you just did.
2. **The OpenAI format is the interface, not the vendor.** vLLM, Ollama, TGI, Groq and LiteLLM all speak it, so your client code outlives any one backend.
3. **Streaming is SSE.** An open connection and a sequence of `data:` lines. Nothing about it is specific to LLMs.
4. **The proxy is not the bottleneck; the GPU is.** Once a model runs behind your server, batching and KV cache memory decide how many users you can serve. That is vLLM's job.
5. **Only the wrapper changes in production.** The tunnel becomes a domain, the subprocess becomes a container, OpenAI becomes vLLM. `server.py` stays.

## Stretch goals

1. **Request logging.** Add an `@app.middleware("http")` that logs a timestamp, the prompt's character count and the latency of every request.
2. **Check the key.** Make `chat_completions` return `401` unless the `Authorization: Bearer ...` header matches a key you choose. Then try B2 with the wrong key.
3. **A metrics route.** Add `GET /metrics` returning the request count, average latency and the last error.
4. **Point the server at Groq.** In cell A2's `env`, set `BACKEND_BASE_URL` to `https://api.groq.com/openai/v1`, `BACKEND_API_KEY` to your Groq key and `DEFAULT_MODEL` to `openai/gpt-oss-20b`. Run cleanup, relaunch, and re-run Part B unchanged.
5. **The ready-made version.** `uv pip install "litellm[proxy]"`, then `litellm --model openai/gpt-4o-mini`. One command gives you an OpenAI-compatible proxy with logging, retries and many backends. Compare it with what you wrote.

## Next

[Lab 6 — RAG Pipeline](../06_RAG_Pipeline/README.md), still on CPU. You retrieve course text with MiniLM and Chroma (Lab 0's vector store, grown up), then generate with `gpt-4o-mini`. The notebook calls OpenAI directly so the retrieval lesson stays in view, but every generation call in it could go through the server you just built.